In [ ]:
# ! rm -r /content/cardiffnlp
#

In [ ]:
# !wget https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/main/data/arabic/test.jsonl .
# !wget https://huggingface.co/datasets/imsoumyaneel/sentiment-analysis-llama2/resolve/main/train.csv content/drive/MyDrive/NLP/datasets
# !wget https://huggingface.co/datasets/CATIE-AQ/allocine_fr_prompt_sentiment_analysis/resolve/main/test.parquet /content/drive/MyDrive/NLP/datasets/

In [ ]:
!pip install sentencepiece


In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
from transformers import pipeline
from scipy.special import softmax
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report



In [ ]:
MODEL =  f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
NUM = 500
FILENAME = '/content/drive/MyDrive/NLP/results.csv'

In [ ]:
def create_confusion_matrix(true_values, predicted_values):
  return confusion_matrix(true_values, predicted_values)

# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)
def plot_data(cm):
  # Create a heatmap with Matplotlib
  plt.matshow(cm, cmap=plt.cm.Blues)
  plt.title("Confusion Matrix")
  plt.colorbar()
  plt.xlabel("Predicted labels")
  plt.ylabel("True labels")
  plt.show()

In [ ]:
data_ar= pd.read_json("/content/drive/MyDrive/NLP/sentiment_analysis/datasets/tweet_sentiment_multilingual_ar_test.jsonl", lines=True)
data_en= pd.read_json("/content/drive/MyDrive/NLP/sentiment_analysis/datasets/tweet_sentiment_multilingual_en_test.jsonl", lines=True)
data_ar.head(10)

,text,label
0,نوال الزغبي (الشاب خالد ليس عالمي) هههههههه أت...,0
1,تقول نوال الزغبي : http,1
2,نوال الزغبي لطيفه الفنانه الوحيده اللي كل الفي...,2
3,لما قالت نوال الزغبي لابقلها هاللقب فرحوا فانز...,0
4,الفنانة نوال الزغبي سنة 90 http,1
5,"@user تذكرني بأغنية نوال الزغبي ""عينيك كدابين""",2
6,بلا تشفير- أمل حمادي بتنتقد النجمة نوال الزغبي...,0
7,“إيغيل فيلمز” تطلق “#ولعانة”.. و #نوال_الزغبي ...,1
8,فنانة لبنانية كبيرة صوتها إسطوري ؟! #ماجدة_ال...,2
9,#لبناني_يقيم_دوره_مكياج_بالرياضمانكبنى غير برا...,0


In [ ]:
data_ar.label.unique()

array([0, 1, 2])

In [ ]:
from transformers import TextClassificationPipeline
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
sentiment_task = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=True)
sentiment_task("نوال الزغبي (الشاب خالد ليس عالمي) هههههههه أتفرجي على ها الفيديو يا مبتدئة http vía @user")

[[{'label': 'negative', 'score': 0.14869169890880585},
  {'label': 'neutral', 'score': 0.5848761796951294},
  {'label': 'positive', 'score': 0.2664320766925812}]]

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax

MODEL =  f"cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
# PT
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)
def sentiment_analysis(text):
  text = preprocess(text)
  # print(text)
  encoded_input = tokenizer(text, return_tensors='pt')
  output = model(**encoded_input)
  scores = output[0][0].detach().numpy()
  scores = softmax(scores)

  return {
      "sentence": text,
      "sentiment": {
        "negative":scores[0],
        "neutral":scores[1] ,
        "positive":scores[2]
      }
    }
sentiment_analysis("نوال الزغبي (الشاب خالد ليس عالمي) هههههههه أتفرجي على ها الفيديو يا مبتدئة http vía @user")

{'sentence': 'نوال الزغبي (الشاب خالد ليس عالمي) هههههههه أتفرجي على ها الفيديو يا مبتدئة http vía @user',
 'sentiment': {'negative': 0.1486917,
  'neutral': 0.5848762,
  'positive': 0.26643208}}

In [ ]:
def sentiment_analysis(text):
  text = preprocess(text)
  # print(text)
  encoded_input = tokenizer(text, return_tensors='pt')
  output = model(**encoded_input)
  scores = output[0][0].detach().numpy()
  scores = softmax(scores)

  return {
      "sentence": text,
      "sentiment": {
        "negative":scores[0],
        "neutral":scores[1] ,
        "positive":scores[2]
      }
    }
def start_process(data):
  results  = []

  for i, text in enumerate (data.text[:NUM]):
    results.append(sentiment_analysis(text))
  return results


  # # y_predicted = [np.argmax(i) for i in results]
  # cm= create_confusion_matrix(data.label[:NUM],results)
  # print(cm)
  # plot_data(cm)
  # # show report
  # report = classification_report(results, data.label[:NUM])
  # print(report)


In [ ]:
start_process(data_ar[:5])
# start_process(data_en)

[{'sentence': 'نوال الزغبي (الشاب خالد ليس عالمي) هههههههه أتفرجي على ها الفيديو يا مبتدئة http vía @user',
  'sentiment': {'negative': 0.1486917,
   'neutral': 0.5848762,
   'positive': 0.26643208}},
 {'sentence': 'تقول نوال الزغبي : http',
  'sentiment': {'negative': 0.058281146,
   'neutral': 0.8698224,
   'positive': 0.07189649}},
 {'sentence': 'نوال الزغبي لطيفه الفنانه الوحيده اللي كل الفيديو كليبات تبعها ماتسبب تلوث بصري ولا سمعي لو صوتها اقل من عادي',
  'sentiment': {'negative': 0.05392249,
   'neutral': 0.19040973,
   'positive': 0.75566775}},
 {'sentence': 'لما قالت نوال الزغبي لابقلها هاللقب فرحوا فانزها 😂😂😂كان لازم ياخدوها اهانة مش ثناء http',
  'sentiment': {'negative': 0.45579585,
   'neutral': 0.27333975,
   'positive': 0.2708644}},
 {'sentence': 'الفنانة نوال الزغبي سنة 90 http',
  'sentiment': {'negative': 0.038318224,
   'neutral': 0.8557077,
   'positive': 0.10597413}}]